# Conjunto de datos completo sin clusterización

In [175]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [176]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [177]:
datos_dia = datos[datos["Cluster KMeans"] == "Soleado"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
35,2022-09-02 11:00:00,17036.043251,20,0,71,2,2,14,11,Soleado,Nublado,5030.740421,22189.147406
36,2022-09-02 12:00:00,27523.885172,21,0,57,4,2,13,12,Soleado,Nublado,17036.043251,29196.986647
37,2022-09-02 13:00:00,20596.278869,23,0,45,5,2,11,13,Soleado,Soleado,27523.885172,25478.471342
38,2022-09-02 14:00:00,28500.000000,24,3,37,6,2,9,14,Soleado,Lluvioso,20596.278869,29057.585772
64,2022-09-03 16:00:00,23122.803757,25,16,41,6,3,11,16,Soleado,Lluvioso,25500.000000,25500.000000
85,2022-09-04 13:00:00,27000.000000,22,32,58,10,2,13,13,Soleado,Lluvioso,22850.891263,20400.000000
86,2022-09-04 14:00:00,25168.164594,23,25,51,12,1,12,14,Soleado,Lluvioso,27000.000000,21548.984244
87,2022-09-04 15:00:00,24300.000000,23,12,47,10,1,11,15,Soleado,Lluvioso,25168.164594,25500.000000
88,2022-09-04 16:00:00,19686.387416,24,12,46,7,1,12,16,Soleado,Lluvioso,24300.000000,23122.803757
89,2022-09-04 17:00:00,22006.958065,25,12,47,5,1,12,17,Soleado,Lluvioso,19686.387416,25602.778606


In [178]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [179]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,20,0,71,2,2,14,11,5030.740421,22189.147406
36,21,0,57,4,2,13,12,17036.043251,29196.986647
37,23,0,45,5,2,11,13,27523.885172,25478.471342
38,24,3,37,6,2,9,14,20596.278869,29057.585772
64,25,16,41,6,3,11,16,25500.000000,25500.000000
...,...,...,...,...,...,...,...,...,...
18282,26,0,32,2,1,8,17,25386.000000,22664.000000
18283,25,0,33,1,1,8,18,22872.000000,15736.000000
18284,23,0,38,0,1,8,19,15825.000000,1407.000000
18285,22,0,45,0,1,9,20,1450.000000,0.000000


In [180]:
y = datos_dia[['Generación']]
y

,Generación
35,17036.043251
36,27523.885172
37,20596.278869
38,28500.000000
64,23122.803757
...,...
18282,22872.000000
18283,15825.000000
18284,1450.000000
18285,0.000000


Dividimos entrenamiento, validación y prueba

In [181]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [182]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4873, y_train: 4873
X_val: 1044, y_val: 1044
X_test: 1045, y_test: 1045


## Escalar con MinMaxScaler

In [183]:
from sklearn.preprocessing import MinMaxScaler

In [184]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [185]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[0.52631579 0.         0.69473684 ... 0.33333333 0.16769135 0.73963825]
 [0.55263158 0.         0.54736842 ... 0.4        0.56786811 0.97323289]
 [0.60526316 0.         0.42105263 ... 0.46666667 0.91746284 0.84928238]
 ...
 [0.21052632 0.         0.48421053 ... 0.06666667 0.         0.        ]
 [0.18421053 0.         0.50526316 ... 0.13333333 0.         0.02536667]
 [0.15789474 0.         0.53684211 ... 0.2        0.01276667 1.        ]]
(4873, 9)


In [186]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.526316,0.000000,0.694737,0.142857,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.552632,0.000000,0.547368,0.285714,0.333333,0.722222,0.400000,0.567868,0.973233
37,0.605263,0.000000,0.421053,0.357143,0.333333,0.611111,0.466667,0.917463,0.849282
38,0.631579,0.061224,0.336842,0.428571,0.333333,0.500000,0.533333,0.686543,0.968586
64,0.657895,0.326531,0.378947,0.428571,0.666667,0.611111,0.666667,0.850000,0.850000
...,...,...,...,...,...,...,...,...,...
12570,0.763158,0.000000,0.084211,0.142857,0.666667,0.111111,0.733333,1.000000,1.000000
12583,0.210526,0.000000,0.505263,0.000000,0.000000,0.055556,0.000000,0.000000,0.000000
12584,0.210526,0.000000,0.484211,0.000000,0.000000,0.111111,0.066667,0.000000,0.000000
12585,0.184211,0.000000,0.505263,0.000000,0.333333,0.111111,0.133333,0.000000,0.025367


In [187]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.31578947 0.         0.37894737 ... 0.26666667 0.29363333 1.        ]
 [0.42105263 0.         0.27368421 ... 0.33333333 0.6314     1.        ]
 [0.52631579 0.         0.21052632 ... 0.4        0.67536667 1.        ]
 ...
 [0.92105263 0.         0.14736842 ... 0.6        0.9326     0.97063333]
 [0.94736842 0.         0.11578947 ... 0.66666667 0.97063333 0.95473333]
 [0.97368421 0.         0.10526316 ... 0.73333333 0.95686667 0.8543    ]]
(1044, 9)


In [188]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
12587,0.315789,0.0,0.378947,0.214286,0.333333,0.055556,0.266667,0.293633,1.000000
12588,0.421053,0.0,0.273684,0.285714,0.333333,0.055556,0.333333,0.631400,1.000000
12589,0.526316,0.0,0.210526,0.357143,0.333333,0.000000,0.400000,0.675367,1.000000
12590,0.605263,0.0,0.178947,0.428571,0.333333,0.055556,0.466667,0.656667,1.000000
12591,0.657895,0.0,0.147368,0.357143,0.333333,0.000000,0.533333,0.713833,1.000000
...,...,...,...,...,...,...,...,...,...
15038,0.842105,0.0,0.210526,1.000000,0.000000,0.555556,0.466667,0.936900,0.928933
15039,0.868421,0.0,0.178947,0.857143,0.000000,0.500000,0.533333,0.928933,0.932600
15040,0.921053,0.0,0.147368,0.642857,0.000000,0.444444,0.600000,0.932600,0.970633
15041,0.947368,0.0,0.115789,0.357143,0.000000,0.388889,0.666667,0.970633,0.954733


In [189]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.94736842 0.         0.10526316 ... 0.8        0.85466667 0.71996667]
 [0.92105263 0.         0.11578947 ... 0.86666667 0.72766667 0.2737    ]
 [0.86842105 0.         0.14736842 ... 0.93333333 0.2827     0.0095    ]
 ...
 [0.60526316 0.         0.34736842 ... 0.86666667 0.5275     0.0469    ]
 [0.57894737 0.         0.42105263 ... 0.93333333 0.04833333 0.        ]
 [0.52631579 0.         0.51578947 ... 1.         0.         0.        ]]
(1045, 9)


In [190]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
15043,0.947368,0.0,0.105263,0.142857,0.333333,0.333333,0.800000,0.854667,0.719967
15044,0.921053,0.0,0.115789,0.071429,0.333333,0.277778,0.866667,0.727667,0.273700
15045,0.868421,0.0,0.147368,0.000000,0.333333,0.388889,0.933333,0.282700,0.009500
15046,0.789474,0.0,0.263158,0.000000,0.000000,0.611111,1.000000,0.010167,0.000000
15055,0.500000,0.0,0.852632,0.000000,0.000000,0.888889,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...
18282,0.684211,0.0,0.284211,0.142857,0.000000,0.444444,0.733333,0.846200,0.755467
18283,0.657895,0.0,0.294737,0.071429,0.000000,0.444444,0.800000,0.762400,0.524533
18284,0.605263,0.0,0.347368,0.000000,0.000000,0.444444,0.866667,0.527500,0.046900
18285,0.578947,0.0,0.421053,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [191]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [192]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[0.51282051 0.         0.70103093 ... 0.33333333 0.16769135 0.73963825]
 [0.53846154 0.         0.55670103 ... 0.4        0.56786811 0.97323289]
 [0.58974359 0.         0.43298969 ... 0.46666667 0.91746284 0.84928238]
 ...
 [0.58974359 0.         0.36082474 ... 0.86666667 0.5275     0.0469    ]
 [0.56410256 0.         0.43298969 ... 0.93333333 0.04833333 0.        ]
 [0.51282051 0.         0.5257732  ... 1.         0.         0.        ]]
(6962, 9)


In [193]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
35,0.512821,0.000000,0.701031,0.142857,0.333333,0.777778,0.333333,0.167691,0.739638
36,0.538462,0.000000,0.556701,0.285714,0.333333,0.722222,0.400000,0.567868,0.973233
37,0.589744,0.000000,0.432990,0.357143,0.333333,0.611111,0.466667,0.917463,0.849282
38,0.615385,0.058824,0.350515,0.428571,0.333333,0.500000,0.533333,0.686543,0.968586
64,0.641026,0.313725,0.391753,0.428571,0.666667,0.611111,0.666667,0.850000,0.850000
...,...,...,...,...,...,...,...,...,...
18282,0.666667,0.000000,0.298969,0.142857,0.000000,0.444444,0.733333,0.846200,0.755467
18283,0.641026,0.000000,0.309278,0.071429,0.000000,0.444444,0.800000,0.762400,0.524533
18284,0.589744,0.000000,0.360825,0.000000,0.000000,0.444444,0.866667,0.527500,0.046900
18285,0.564103,0.000000,0.432990,0.000000,0.000000,0.500000,0.933333,0.048333,0.000000


In [194]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [195]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.68654263]
 ...
 [0.        ]
 [0.01276667]
 [0.29363333]]
(4873, 1)


In [196]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
35,0.567868
36,0.917463
37,0.686543
38,0.950000
64,0.770760
...,...
12570,1.000000
12583,0.000000
12584,0.000000
12585,0.012767


In [197]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.6314    ]
 [0.67536667]
 [0.65666667]
 ...
 [0.97063333]
 [0.95686667]
 [0.85466667]]
(1044, 1)


In [198]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
12587,0.631400
12588,0.675367
12589,0.656667
12590,0.713833
12591,0.627167
...,...
15038,0.928933
15039,0.932600
15040,0.970633
15041,0.956867


In [199]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[0.72766667]
 [0.2827    ]
 [0.01016667]
 ...
 [0.04833333]
 [0.        ]
 [0.        ]]
(1045, 1)


In [200]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
15043,0.727667
15044,0.282700
15045,0.010167
15046,0.000000
15055,0.000000
...,...
18282,0.762400
18283,0.527500
18284,0.048333
18285,0.000000


In [201]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [202]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.56786811]
 [0.91746284]
 [0.68654263]
 ...
 [0.04833333]
 [0.        ]
 [0.        ]]
(6962, 1)


In [203]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
35,0.567868
36,0.917463
37,0.686543
38,0.950000
64,0.770760
...,...
18282,0.762400
18283,0.527500
18284,0.048333
18285,0.000000


## Preparación para Redes Neuronales

In [204]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [205]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [206]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4825, 48, 9), y_train: (4825, 1)
X_val: (996, 48, 9), y_val: (996, 1)
X_test: (997, 48, 9), y_test: (997, 1)


## Optuna

In [207]:
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

### Optimización de LightGBM

In [ ]:
import optuna
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error

# Definir la función de optimización
def objective(trial):
    # Espacio de búsqueda para los hiperparámetros
    num_leaves = trial.suggest_int('num_leaves', 10, 1000)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.1, 1.0)
    min_data_in_leaf = trial.suggest_int('min_data_in_leaf', 10, 100)

    # Modelo LightGBM con los hiperparámetros sugeridos
    model = LGBMRegressor(
        num_leaves=num_leaves,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_data_in_leaf=min_data_in_leaf,
        random_state=42,
        device= gpu_support
    )

    # Entrenamos el modelo con los datos de entrenamiento
    model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Ejecutar la optimización con 50 iteraciones
LightGBM_study = optuna.create_study(direction="minimize")
LightGBM_study.optimize(objective, n_trials=50)

# Obtener los mejores hiperparámetros
best_params = LightGBM_study.best_params
print("Mejores hiperparámetros:", best_params)

[I 2025-03-11 19:23:14,318] A new study created in memory with name: no-name-4d9c024d-f025-4b94-85a2-5fae344fd3ce
[I 2025-03-11 19:23:15,374] Trial 0 finished with value: 0.006669897224094316 and parameters: {'num_leaves': 440, 'subsample': 0.250625180763996, 'colsample_bytree': 0.452361707656043, 'min_data_in_leaf': 33}. Best is trial 0 with value: 0.006669897224094316.
[I 2025-03-11 19:23:15,655] Trial 1 finished with value: 0.005384780394145866 and parameters: {'num_leaves': 524, 'subsample': 0.3587505573575017, 'colsample_bytree': 0.9803850229727685, 'min_data_in_leaf': 84}. Best is trial 1 with value: 0.005384780394145866.
[I 2025-03-11 19:23:16,483] Trial 2 finished with value: 0.008062598285722191 and parameters: {'num_leaves': 206, 'subsample': 0.8495146451658534, 'colsample_bytree': 0.525648714219341, 'min_data_in_leaf': 12}. Best is trial 1 with value: 0.005384780394145866.
[I 2025-03-11 19:23:17,324] Trial 3 finished with value: 0.006809442571635958 and parameters: {'num_lea

Mejores hiperparámetros: {'num_leaves': 11, 'subsample': 0.7195971223154776, 'colsample_bytree': 0.9869895068945519, 'min_data_in_leaf': 100}


### Random Forest

In [212]:
def objective(trial):
    # Definir los hiperparámetros a optimizar
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 10, 50, step=5)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    # Definir el modelo con los hiperparámetros actuales
    RF_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        criterion="squared_error",
        random_state=0
    )

    # Entrenar el modelo
    RF_model.fit(X_train_scaled_df, y_train_scaled_df)

    # Predicciones en el conjunto de validación
    y_pred = RF_model.predict(X_val_scaled_df)
    rmse = mean_squared_error(y_val_scaled_df, y_pred)

    return rmse  # Optuna minimizará este valor

# Crear el estudio de optimización
RF_study = optuna.create_study(direction="minimize")
RF_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", RF_study.best_params)

[I 2025-03-11 19:25:57,467] A new study created in memory with name: no-name-929e11b1-6e85-48ae-adf7-933b9fc550cb
[I 2025-03-11 19:26:13,962] Trial 0 finished with value: 0.011639761189434815 and parameters: {'n_estimators': 300, 'max_depth': 35, 'min_samples_split': 15, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 0 with value: 0.011639761189434815.
[I 2025-03-11 19:26:24,255] Trial 1 finished with value: 0.009097836418609567 and parameters: {'n_estimators': 250, 'max_depth': 40, 'min_samples_split': 13, 'min_samples_leaf': 8, 'bootstrap': False}. Best is trial 1 with value: 0.009097836418609567.
[I 2025-03-11 19:26:38,958] Trial 2 finished with value: 0.011574890872546045 and parameters: {'n_estimators': 300, 'max_depth': 25, 'min_samples_split': 18, 'min_samples_leaf': 2, 'bootstrap': False}. Best is trial 1 with value: 0.009097836418609567.
[I 2025-03-11 19:26:59,495] Trial 3 finished with value: 0.012045692846609326 and parameters: {'n_estimators': 400, 'max_depth': 3

Mejores hiperparámetros: {'n_estimators': 400, 'max_depth': 45, 'min_samples_split': 17, 'min_samples_leaf': 10, 'bootstrap': True}


### CTNET

In [213]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

# Construir modelo con hiperparámetros sugeridos
def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout, mlp_dropout):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs

    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    
    for units in mlp_units:
        x = layers.Dense(units, activation="relu")(x)

    x = layers.Dropout(mlp_dropout)(x)
    outputs = layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs)

In [214]:
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [ ]:
# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    head_size = trial.suggest_int("head_size", 2, 8)
    num_heads = trial.suggest_int("num_heads", 2, 8)
    ff_dim = trial.suggest_int("ff_dim", 16, 128, step=16)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 5)
    mlp_units = [trial.suggest_int("mlp_units_1", 64, 512, step=64),
                 trial.suggest_int("mlp_units_2", 32, 256, step=32)]
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Construir el modelo
    model = build_model(
        input_shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]),
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar y entrenar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Minimizar MSE

# Crear el estudio de optimización
CTNET_study = optuna.create_study(direction="minimize")
CTNET_study.optimize(objective, n_trials=50)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", CTNET_study.best_params)


[I 2025-03-11 19:35:38,750] A new study created in memory with name: no-name-49fbce82-cf33-4436-b2f8-44cb7b02117d
[I 2025-03-11 20:01:56,672] Trial 0 finished with value: 0.11877622455358505 and parameters: {'head_size': 4, 'num_heads': 8, 'ff_dim': 112, 'num_transformer_blocks': 4, 'mlp_units_1': 320, 'mlp_units_2': 160, 'dropout': 0.23262113635016815, 'mlp_dropout': 0.25004517288331674, 'learning_rate': 2.00581114844137e-05, 'batch_size': 512}. Best is trial 0 with value: 0.11877622455358505.
[I 2025-03-11 20:23:06,728] Trial 1 finished with value: 0.04681694507598877 and parameters: {'head_size': 6, 'num_heads': 7, 'ff_dim': 80, 'num_transformer_blocks': 2, 'mlp_units_1': 448, 'mlp_units_2': 256, 'dropout': 0.303020430244923, 'mlp_dropout': 0.2502938239162138, 'learning_rate': 0.00013214377235462986, 'batch_size': 512}. Best is trial 1 with value: 0.04681694507598877.
[I 2025-03-11 20:38:52,645] Trial 2 finished with value: 0.03452784940600395 and parameters: {'head_size': 5, 'num_h

### Forescasting

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Conv1D, BatchNormalization, MaxPooling1D, Bidirectional, LSTM, Dropout, Dense
from tensorflow.keras.optimizers import Adam

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters = trial.suggest_categorical("filters", [32, 64, 128])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    lstm_units_1 = trial.suggest_categorical("lstm_units_1", [64, 128, 256])
    lstm_units_2 = trial.suggest_categorical("lstm_units_2", [32, 64, 128])
    lstm_units_3 = trial.suggest_categorical("lstm_units_3", [16, 32, 64])
    dropout_lstm = trial.suggest_float("dropout_lstm", 0.1, 0.5)
    dropout_dense = trial.suggest_float("dropout_dense", 0.1, 0.5)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])  # 🔹 Optimización del batch size

    # Construcción del modelo
    model = Sequential()
    model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

    # CNN
    model.add(Conv1D(filters=filters, kernel_size=kernel_size, padding='same', activation='relu'))
    model.add(BatchNormalization())  
    model.add(MaxPooling1D(pool_size=2))

    # BiLSTM
    model.add(Bidirectional(LSTM(lstm_units_1, return_sequences=True)))
    model.add(Bidirectional(LSTM(lstm_units_2, return_sequences=True)))
    model.add(Dropout(dropout_lstm))
    model.add(Bidirectional(LSTM(lstm_units_3, return_sequences=False)))

    # Normalización y Dropout
    model.add(BatchNormalization())
    model.add(Dropout(dropout_dense))

    # Capas Densas
    model.add(Dense(16, activation='relu'))
    model.add(Dense(1, activation='relu'))

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenar el modelo con validación
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se puede detener antes con Early Stopping
        batch_size=batch_size,  # 🔹 Batch size optimizado
        validation_data=(X_val_windowed, y_val_windowed),  # 🔹 Validación en cada época
        callbacks=[early_stop],  # 🔹 Early Stopping activado
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Forecasting_study = optuna.create_study(direction="minimize")
Forecasting_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Forecasting_study.best_params)

### Photovoltaic

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Bidirectional, GRU, MultiHeadAttention
from tensorflow.keras.layers import Flatten, Dropout, Dense
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import mean_squared_error

# Definir la función objetivo para Optuna
def objective(trial):
    # Hiperparámetros a optimizar
    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 128])
    filters_2 = trial.suggest_categorical("filters_2", [64, 128, 256])
    kernel_size = trial.suggest_int("kernel_size", 2, 5)
    gru_units = trial.suggest_categorical("gru_units", [32, 64, 128])
    num_heads = trial.suggest_categorical("num_heads", [2, 4, 8])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5)
    dense_units_1 = trial.suggest_categorical("dense_units_1", [32, 64, 128])
    dense_units_2 = trial.suggest_categorical("dense_units_2", [16, 32, 64])
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [128, 256, 512])

    # Definir el modelo
    inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

    # Primera capa CNN
    x = Conv1D(filters=filters_1, kernel_size=kernel_size, padding='same', activation='relu')(inputs)
    x = MaxPooling1D(pool_size=2)(x)

    # Segunda capa CNN
    x = Conv1D(filters=filters_2, kernel_size=kernel_size, padding='same', activation='relu')(x)
    x = MaxPooling1D(pool_size=2)(x)

    # Capa BiGRU
    x = Bidirectional(GRU(gru_units, return_sequences=True))(x)

    # Atención MultiHead
    attention = MultiHeadAttention(num_heads=num_heads, key_dim=128)(x, x)

    # Aplanar y Dropout
    x = Flatten()(attention)
    x = Dropout(dropout_rate)(x)
    x = Dense(dense_units_1, activation="relu", kernel_regularizer=l2(0.01))(x)
    x = Dense(dense_units_2, activation="relu")(x)

    # Capa de salida
    outputs = Dense(1, activation="linear")(x)

    # Construcción del modelo
    model = Model(inputs=inputs, outputs=outputs)

    # Compilar el modelo
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Early Stopping
    early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

    # Entrenar el modelo
    history = model.fit(
        X_train_windowed, y_train_windowed,
        epochs=100,  # Se detendrá con Early Stopping
        batch_size=batch_size,
        validation_data=(X_val_windowed, y_val_windowed),
        callbacks=[early_stop],
        verbose=0
    )

    # Obtener la mejor pérdida de validación
    val_loss = min(history.history['val_loss'])
    return val_loss  # Optuna minimizará este valor

# Crear el estudio de optimización
Photovoltaic_study = optuna.create_study(direction="minimize")
Photovoltaic_study.optimize(objective, n_trials=50, n_jobs=-1)  # Ejecutar 50 pruebas

# Imprimir los mejores hiperparámetros encontrados
print("Mejores hiperparámetros:", Photovoltaic_study.best_params)

## Definición de modelos

### RandomForest

In [118]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [119]:
resultados

,LightGBM
15043,21155.082057
15044,8179.469627
15045,142.771562
15046,0.0
15055,32.18063
...,...
18282,21973.744337
18283,15183.20265
18284,1503.328981
18285,0.0


In [120]:
predicciones = y_test.copy()
predicciones

,Generación
15043,21830.0
15044,8481.0
15045,305.0
15046,0.0
15055,0.0
...,...
18282,22872.0
18283,15825.0
18284,1450.0
18285,0.0


In [121]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
15043,21830.0,21155.082057
15044,8481.0,8179.469627
15045,305.0,142.771562
15046,0.0,0.0
15055,0.0,32.18063
...,...,...
18282,22872.0,21973.744337
18283,15825.0,15183.20265
18284,1450.0,1503.328981
18285,0.0,0.0


In [122]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1165.0131
RMSE: 2082.7462
R²: 0.9600


## Random Forest

In [123]:
from sklearn.ensemble import RandomForestRegressor

In [124]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [125]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
15043,21830.0,21155.082057,21474.980726
15044,8481.0,8179.469627,10113.609782
15045,305.0,142.771562,332.6175
15046,0.0,0.0,0.0
15055,0.0,32.18063,0.0325
...,...,...,...
18282,22872.0,21973.744337,22272.162014
18283,15825.0,15183.20265,16869.890859
18284,1450.0,1503.328981,1679.507845
18285,0.0,0.0,0.0


## Preparación redes neuronales

## CTNET

In [129]:
import tensorflow as tf
from tensorflow.keras import layers

In [130]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [131]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [132]:
history = compile_and_fit(CTNET)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 79s 3s/step - loss: 0.4600 - mean_absolute_error: 0.5620 - mean_absolute_percentage_error: 368025.6562 - root_mean_squared_error: 0.6782 - val_loss: 0.3812 - val_mean_absolute_error: 0.5009 - val_mean_absolute_percentage_error: 2722940.2500 - val_root_mean_squared_error: 0.6174
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - loss: 0.4489 - mean_absolute_error: 0.5563 - mean_absolute_percentage_error: 2231250.7500 - root_mean_squared_error: 0.6700 - val_loss: 0.3650 - val_mean_absolute_error: 0.4923 - val_mean_absolute_percentage_error: 6162399.5000 - val_root_mean_squared_error: 0.6042
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 26s 2s/step - loss: 0.4294 - mean_absolute_error: 0.5449 - mean_absolute_percentage_error: 4447563.5000 - root_mean_squared_error: 0.6553 - val_loss: 0.3443 - val_mean_absolute_error: 0.4816 - val_mean_absolute_percentage_error: 10723579.0000 - val_root_mean_squared_error: 0.5868
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 13s 1s/step - lo

In [133]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 181ms/step


array([[0.15496975],
       [0.14259696],
       [0.17061563],
       [0.33523908],
       [0.51664966],
       [0.7786477 ],
       [0.90447944],
       [0.95244294],
       [0.9495289 ],
       [0.9263549 ],
       [0.91107476],
       [0.8768217 ],
       [0.828679  ],
       [0.7551853 ],
       [0.43288106],
       [0.16479386],
       [0.1294194 ],
       [0.13756981],
       [0.18890068],
       [0.34198895],
       [0.5161552 ],
       [0.7638336 ],
       [0.8760454 ],
       [0.92371327],
       [0.93901384],
       [0.9322058 ],
       [0.9220883 ],
       [0.8465508 ],
       [0.5826834 ],
       [0.23640178],
       [0.11141685],
       [0.12764807],
       [0.1657343 ],
       [0.19806947],
       [0.37106028],
       [0.49522305],
       [0.74896604],
       [0.8740291 ],
       [0.92795473],
       [0.92086667],
       [0.90233177],
       [0.87671286],
       [0.85705376],
       [0.81295604],
       [0.72684675],
       [0.39011842],
       [0.15806022],
       [0.132

In [134]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [135]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [136]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15043,21830.0,21155.082057,21474.980726,NaN
15044,8481.0,8179.469627,10113.609782,NaN
15045,305.0,142.771562,332.6175,NaN
15046,0.0,0.0,0.0,NaN
15055,0.0,32.18063,0.0325,NaN
...,...,...,...,...
18282,22872.0,21973.744337,22272.162014,27485.386719
18283,15825.0,15183.20265,16869.890859,19769.083984
18284,1450.0,1503.328981,1679.507845,8603.554688
18285,0.0,0.0,0.0,7298.136230


In [137]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [ ]:
import optuna
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split

# Definir la función objetivo para Optuna
def objective(trial):
    # Sugerir valores para los hiperparámetros
    head_size = trial.suggest_int("head_size", 8, 64, step=8)
    num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
    ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
    num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
    mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
    dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
    mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
    learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

    # Construcción del modelo con los hiperparámetros sugeridos
    model = build_model(
        input_shape=X_train_windowed.shape[1:],
        head_size=head_size,
        num_heads=num_heads,
        ff_dim=ff_dim,
        num_transformer_blocks=num_transformer_blocks,
        mlp_units=mlp_units,
        dropout=dropout,
        mlp_dropout=mlp_dropout
    )

    # Compilar el modelo con los hiperparámetros sugeridos
    model.compile(
        loss=tf.keras.losses.MeanSquaredError(),
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )

    # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
    history = model.fit(
        X_train_windowed, y_train_windowed,
        validation_split=0.2,
        epochs=50,  # Reducimos las épocas para acelerar la búsqueda
        batch_size=512,
        verbose=0
    )

    # Obtener la métrica de validación (RMSE) y minimizarla
    val_rmse = min(history.history["val_root_mean_squared_error"])
    
    return val_rmse  # Queremos minimizar el RMSE

# Ejecutar la optimización de hiperparámetros
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20, timeout=36000)  # 20 iteraciones, máximo 1 hora

# Mostrar los mejores hiperparámetros encontrados
best_params = study.best_params
print(f"Mejores hiperparámetros: {best_params}")


In [139]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [140]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 202s 6s/step - loss: 0.4181 - mean_absolute_error: 0.5338 - mean_absolute_percentage_error: 5166416.5000 - root_mean_squared_error: 0.6462 - val_loss: 0.1810 - val_mean_absolute_error: 0.3869 - val_mean_absolute_percentage_error: 61192448.0000 - val_root_mean_squared_error: 0.4255
Epoch 2/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 51s 6s/step - loss: 0.1889 - mean_absolute_error: 0.3816 - mean_absolute_percentage_error: 59043684.0000 - root_mean_squared_error: 0.4344 - val_loss: 0.1575 - val_mean_absolute_error: 0.3226 - val_mean_absolute_percentage_error: 134880624.0000 - val_root_mean_squared_error: 0.3968
Epoch 3/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 81s 6s/step - loss: 0.1567 - mean_absolute_error: 0.3547 - mean_absolute_percentage_error: 76038232.0000 - root_mean_squared_error: 0.3958 - val_loss: 0.1386 - val_mean_absolute_error: 0.3430 - val_mean_absolute_percentage_error: 94136000.0000 - val_root_mean_squared_error: 0.3723
Epoch 4/50
8/8 ━━━━━━━━━━━━━━━━━━━━ 38s 5s/st

In [141]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

32/32 ━━━━━━━━━━━━━━━━━━━━ 41s 641ms/step


array([[ 0.0536165 ],
       [-0.01127326],
       [ 0.04021256],
       [ 0.09968519],
       [ 0.51191264],
       [ 0.7897858 ],
       [ 0.90980977],
       [ 0.91246146],
       [ 0.9586    ],
       [ 0.9226785 ],
       [ 0.9325285 ],
       [ 0.9432946 ],
       [ 0.9536638 ],
       [ 0.88629097],
       [ 0.74278545],
       [ 0.42130703],
       [ 0.00941055],
       [-0.02389687],
       [ 0.12105928],
       [ 0.0707441 ],
       [ 0.5090988 ],
       [ 0.8096711 ],
       [ 0.90799564],
       [ 0.91653955],
       [ 0.94543356],
       [ 0.9293543 ],
       [ 0.9364411 ],
       [ 0.7914521 ],
       [ 0.6741058 ],
       [ 0.46499878],
       [ 0.14402826],
       [ 0.04253279],
       [ 0.10911013],
       [ 0.19314337],
       [ 0.07748315],
       [ 0.4378335 ],
       [ 0.7551177 ],
       [ 0.8664478 ],
       [ 0.91868025],
       [ 0.90551203],
       [ 0.9176829 ],
       [ 0.93874735],
       [ 0.93326384],
       [ 0.90519726],
       [ 0.8161209 ],
       [ 0

In [142]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [143]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [144]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
15043,21830.0,21155.082057,21474.980726,NaN
15044,8481.0,8179.469627,10113.609782,NaN
15045,305.0,142.771562,332.6175,NaN
15046,0.0,0.0,0.0,NaN
15055,0.0,32.18063,0.0325,NaN
...,...,...,...,...
18282,22872.0,21973.744337,22272.162014,23384.568359
18283,15825.0,15183.20265,16869.890859,11076.665039
18284,1450.0,1503.328981,1679.507845,1510.602661
18285,0.0,0.0,0.0,2154.357666


## Forecasting

In [145]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [146]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_24 (Conv1D)              │ (None, 48, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,985 (1.55 MB)

 Trainable params: 405,729 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [147]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [148]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 728s 512ms/step - loss: 0.1794 - mae: 0.4793 - val_loss: 0.0504 - val_mae: 0.2332
Epoch 2/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 235s 379ms/step - loss: 0.1015 - mae: 0.3431 - val_loss: 0.0406 - val_mae: 0.2209
Epoch 3/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 305s 442ms/step - loss: 0.0815 - mae: 0.3053 - val_loss: 0.0316 - val_mae: 0.1874
Epoch 4/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 341s 463ms/step - loss: 0.0695 - mae: 0.2806 - val_loss: 0.0215 - val_mae: 0.1401
Epoch 5/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 340s 482ms/step - loss: 0.0667 - mae: 0.2748 - val_loss: 0.0227 - val_mae: 0.1403
Epoch 6/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 298s 476ms/step - loss: 0.0578 - mae: 0.2515 - val_loss: 0.0217 - val_mae: 0.1381
Epoch 7/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 339s 492ms/step - loss: 0.0552 - mae: 0.2448 - val_loss: 0.0185 - val_mae: 0.1251
Epoch 8/100
604/604 ━━━━━━━━━━━━━━━━━━━━ 245s 394ms/step - loss: 0.0484 - mae: 0.2277 - val_loss: 0.0187 - val_mae: 0.1280
Epoch 9/100
604/

In [149]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

32/32 ━━━━━━━━━━━━━━━━━━━━ 48s 840ms/step


array([[0.        ],
       [0.        ],
       [0.        ],
       [0.        ],
       [0.42313248],
       [0.7914566 ],
       [0.8077628 ],
       [0.885322  ],
       [0.87751174],
       [0.883356  ],
       [0.86353004],
       [0.8693399 ],
       [0.8681351 ],
       [0.8185429 ],
       [0.6218435 ],
       [0.21578395],
       [0.        ],
       [0.        ],
       [0.24168979],
       [0.24536669],
       [0.43615082],
       [0.73084307],
       [0.7739767 ],
       [0.8546025 ],
       [0.8558662 ],
       [0.85363954],
       [0.8442383 ],
       [0.6867374 ],
       [0.5836442 ],
       [0.21578395],
       [0.18221702],
       [0.        ],
       [0.        ],
       [0.23799986],
       [0.        ],
       [0.30278835],
       [0.75803727],
       [0.77358085],
       [0.87931895],
       [0.8568387 ],
       [0.8698833 ],
       [0.8388461 ],
       [0.8593444 ],
       [0.8545511 ],
       [0.8405014 ],
       [0.7000327 ],
       [0.2288469 ],
       [0.012

In [150]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [151]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [152]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
15043,21830.0,21155.082057,21474.980726,NaN,NaN
15044,8481.0,8179.469627,10113.609782,NaN,NaN
15045,305.0,142.771562,332.6175,NaN,NaN
15046,0.0,0.0,0.0,NaN,NaN
15055,0.0,32.18063,0.0325,NaN,NaN
...,...,...,...,...,...
18282,22872.0,21973.744337,22272.162014,23384.568359,22015.341797
18283,15825.0,15183.20265,16869.890859,11076.665039,18189.736328
18284,1450.0,1503.328981,1679.507845,1510.602661,0.000000
18285,0.0,0.0,0.0,2154.357666,0.000000


## Métricas

In [153]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

15117      512.0
15118        0.0
15127        0.0
15128     2647.0
15129    14094.0
          ...   
18282    22872.0
18283    15825.0
18284     1450.0
18285        0.0
18286        0.0
Name: Generación, Length: 997, dtype: float64

## Photovoltaic

In [154]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_15"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_25 (Conv1D)  │ (None, 48, 64)    │      2,368 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_25[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_26 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_26[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_18 (Dense)    │ (None, 64)        │     98,368 │ dropout_19[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 32)        │      2,080 │ dense_18[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 1)         │         33 │ dense_19[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 474,049 (1.81 MB)

 Trainable params: 474,049 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [155]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [156]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50


302/302 ━━━━━━━━━━━━━━━━━━━━ 210s 218ms/step - loss: 1.1151 - mae: 0.3537 - val_loss: 0.4760 - val_mae: 0.2777
Epoch 2/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 53s 166ms/step - loss: 0.3682 - mae: 0.2342 - val_loss: 0.1626 - val_mae: 0.1595
Epoch 3/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 39s 115ms/step - loss: 0.1446 - mae: 0.1720 - val_loss: 0.0843 - val_mae: 0.1497
Epoch 4/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 32s 79ms/step - loss: 0.0829 - mae: 0.1587 - val_loss: 0.0573 - val_mae: 0.1346
Epoch 5/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 31s 103ms/step - loss: 0.0605 - mae: 0.1470 - val_loss: 0.0472 - val_mae: 0.1384
Epoch 6/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 60s 158ms/step - loss: 0.0507 - mae: 0.1445 - val_loss: 0.0432 - val_mae: 0.1347
Epoch 7/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 83s 148ms/step - loss: 0.0447 - mae: 0.1364 - val_loss: 0.0374 - val_mae: 0.1287
Epoch 8/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 83s 148ms/step - loss: 0.0443 - mae: 0.1419 - val_loss: 0.0363 - val_mae: 0.1240
Epoch 9/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 40s

In [157]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

32/32 ━━━━━━━━━━━━━━━━━━━━ 23s 370ms/step


array([[0.02740759],
       [0.02802015],
       [0.21123044],
       [0.0614012 ],
       [0.5445817 ],
       [0.8819997 ],
       [0.90324944],
       [0.9176143 ],
       [0.94390434],
       [0.94702494],
       [0.96202695],
       [0.922407  ],
       [0.91047204],
       [0.8053058 ],
       [0.5272221 ],
       [0.15767384],
       [0.02883917],
       [0.15499492],
       [0.3849177 ],
       [0.13116814],
       [0.5857817 ],
       [0.9022453 ],
       [0.9085057 ],
       [0.91731006],
       [0.93375516],
       [0.9429768 ],
       [0.9475805 ],
       [0.7650006 ],
       [0.5231183 ],
       [0.26635253],
       [0.11078924],
       [0.02775986],
       [0.04339744],
       [0.2560154 ],
       [0.04985664],
       [0.51142305],
       [0.8352691 ],
       [0.8669221 ],
       [0.92020047],
       [0.8881665 ],
       [0.90927094],
       [0.9035956 ],
       [0.8982045 ],
       [0.88995147],
       [0.85319436],
       [0.64501256],
       [0.19620551],
       [0.033

In [158]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [159]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [160]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
15043,21830.0,21155.082057,21474.980726,NaN,NaN,NaN
15044,8481.0,8179.469627,10113.609782,NaN,NaN,NaN
15045,305.0,142.771562,332.6175,NaN,NaN,NaN
15046,0.0,0.0,0.0,NaN,NaN,NaN
15055,0.0,32.18063,0.0325,NaN,NaN,NaN
...,...,...,...,...,...,...
18282,22872.0,21973.744337,22272.162014,23384.568359,22015.341797,21476.437500
18283,15825.0,15183.20265,16869.890859,11076.665039,18189.736328,15476.440430
18284,1450.0,1503.328981,1679.507845,1510.602661,0.000000,2367.579590
18285,0.0,0.0,0.0,2154.357666,0.000000,871.135620


In [161]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1165.0131
RMSE: 2082.7462
R²: 0.9600
Random Forest
MAE: 1298.5872
RMSE: 2435.8068
R²: 0.9452
CTNET
MAE: 3995.2136
RMSE: 6285.5283
R²: 0.6328
Forecast
MAE: 4294.4490
RMSE: 5879.6775
R²: 0.6787
Photovoltaic
MAE: 3862.2200
RMSE: 5765.6045
R²: 0.6911


In [162]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [163]:
predicciones_train = y_train.copy()

In [164]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

In [165]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [166]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

151/151 ━━━━━━━━━━━━━━━━━━━━ 13s 71ms/step


In [167]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

151/151 ━━━━━━━━━━━━━━━━━━━━ 16s 86ms/step


In [168]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

151/151 ━━━━━━━━━━━━━━━━━━━━ 12s 49ms/step


In [169]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
35,17036.043251,17704.978706,18364.998529,NaN,NaN,NaN
36,27523.885172,22373.712679,25632.981146,NaN,NaN,NaN
37,20596.278869,26215.332190,23382.503307,NaN,NaN,NaN
38,28500.000000,23098.593066,24922.688282,NaN,NaN,NaN
64,23122.803757,23871.764861,23873.501835,NaN,NaN,NaN
...,...,...,...,...,...,...
12570,30000.000000,30000.000000,29905.597500,28192.080078,24095.363281,27167.457031
12583,0.000000,0.000000,0.000000,23977.191406,18742.062500,19598.519531
12584,0.000000,0.000000,0.000000,8328.705078,0.000000,5342.332520
12585,383.000000,974.014240,650.471851,3930.295410,4332.046387,1908.347900


In [170]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 950.3024
RMSE: 1774.4748
R²: 0.9757
Random Forest
MAE: 371.9471
RMSE: 756.2389
R²: 0.9956
CTNET
MAE: 2572.4235
RMSE: 4149.5901
R²: 0.8673
Forecast
MAE: 3495.8807
RMSE: 5469.3679
R²: 0.7694
Photovoltaic
MAE: 3160.2586
RMSE: 4980.4300
R²: 0.8088


In [171]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,...,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [172]:
datos.to_excel("04.2_Predicciones_Conjunto_soleado KMeans.xlsx", index=True)

## Guardamos los modelos

In [173]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_2_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_2_RandomForest_model.pkl")


['4_2_RandomForest_model.pkl']

In [174]:
CTNET.save("4_2_CTNET_model.keras")
Forecast_model.save("4_2_Forecast_model.keras")
Photo_model.save("4_2_Photo_model.keras")